In [1]:
import re
import spacy
import nltk
from nltk.corpus import treebank
from sklearn.metrics import precision_score, recall_score, f1_score

# =========================================================
# INITIALIZATION & DATASET SETUP
# =========================================================
# Uncomment the following line if you need to download the dataset for the first time
# nltk.download('treebank')

print("========== NAMED ENTITY RECOGNITION (NER) ==========")

# Load spaCy pre-trained statistical model
nlp_spacy = spacy.load("en_core_web_sm")

# Load NLTK Tagged Corpus (Treebank) as specified
print("Loading NLTK Treebank corpus...")

# Using the first 3 sentences from the Treebank corpus to form our text data
nltk_sentences = treebank.sents()[:3]

# Flatten the sentences into a single list of words and create a raw text string
words = [word for sent in nltk_sentences for word in sent]
raw_text = " ".join(words)

print(f"\nRaw Text for Analysis:\n{raw_text}\n")


# ---------------------------------------------------------
# GROUND TRUTH DEFINITION FOR EVALUATION
# ---------------------------------------------------------
# To evaluate Precision, Recall, and F1-score, we establish a ground truth.
# We manually label the core entities in these specific Treebank sentences 
# using a binary format (1 if the word is part of an entity, 0 otherwise).
#
# Sent 1: Pierre Vinken , 61 years old , will join the board as a nonexecutive director Nov. 29 .
# Entities: Pierre Vinken (Person)
# Sent 2: Mr. Vinken is chairman of Elsevier N.V. , the Dutch publishing group .
# Entities: Vinken (Person), Elsevier N.V. (Organization), Dutch (Misc)
# Sent 3: Rudolph Agnew , 55 years old and former chairman of Consolidated Gold Fields PLC , was named a nonexecutive director of this British industrial conglomerate .
# Entities: Rudolph Agnew (Person), Consolidated Gold Fields PLC (Organization), British (Misc)

true_entities = ["Pierre", "Vinken", "Elsevier", "N.V.", "Dutch", "Rudolph", "Agnew", "Consolidated", "Gold", "Fields", "PLC", "British"]

# Build the true labels array
y_true = [1 if word in true_entities else 0 for word in words]


# =========================================================
# 1. RULE-BASED APPROACH (PATTERN MATCHING)
# =========================================================
print("========== 1. RULE-BASED NER ==========")
# Using Regular Expressions to find capitalized words (Proper Nouns) as a basic rule for NER.
# This assumes any capitalized word (excluding common titles) is an entity.

rule_based_extracted = []
y_pred_rule = []

for word in words:
    # Pattern: Matches words starting with a capital letter followed by lowercase letters or dots
    if re.match(r'^[A-Z][A-Za-z\.]+$', word) and word not in ["Mr.", "The"]:
        y_pred_rule.append(1)
        rule_based_extracted.append(word)
    else:
        y_pred_rule.append(0)

print(f"Entities extracted via Rule-Based (Regex):")
print(rule_based_extracted)


# =========================================================
# 2. PRE-TRAINED STATISTICAL MODEL (SPACY)
# =========================================================
print("\n========== 2. STATISTICAL NER (spaCy) ==========")
# Process the raw text through the spaCy pipeline
doc = nlp_spacy(raw_text)

spacy_extracted = []
y_pred_spacy = []

print("Entities extracted via spaCy:")
for ent in doc.ents:
    # Extract and classify the named entities into predefined categories
    print(f" - {ent.text} (Class: {ent.label_})")
    # Split multi-word entities to match our token-level evaluation array
    spacy_extracted.extend(ent.text.split())
    
# Build the prediction array for the statistical model
for word in words:
    # Match words to the extracted spaCy entities
    if word in spacy_extracted:
        y_pred_spacy.append(1)
    else:
        y_pred_spacy.append(0)


# =========================================================
# 3. MODEL EVALUATION
# =========================================================
print("\n========== 3. PERFORMANCE EVALUATION ==========")

# Evaluate Rule-Based Approach
rule_precision = precision_score(y_true, y_pred_rule, zero_division=0)
rule_recall = recall_score(y_true, y_pred_rule, zero_division=0)
rule_f1 = f1_score(y_true, y_pred_rule, zero_division=0)

print("--- Rule-Based Approach Metrics ---")
print(f"Precision : {rule_precision:.4f}")
print(f"Recall    : {rule_recall:.4f}")
print(f"F1-Score  : {rule_f1:.4f}")

# Evaluate Statistical Approach (spaCy)
spacy_precision = precision_score(y_true, y_pred_spacy, zero_division=0)
spacy_recall = recall_score(y_true, y_pred_spacy, zero_division=0)
spacy_f1 = f1_score(y_true, y_pred_spacy, zero_division=0)

print("\n--- Statistical Approach (spaCy) Metrics ---")
print(f"Precision : {spacy_precision:.4f}")
print(f"Recall    : {spacy_recall:.4f}")
print(f"F1-Score  : {spacy_f1:.4f}")


# =========================================================
# 4. COMPARISON & JUSTIFICATION
# =========================================================
print("\n========== 4. COMPARISON & JUSTIFICATION ==========")
justification = """
Rule-Based vs. Statistical Comparison:

1. Rule-Based Approach (Regex):
   - Strengths: Extremely fast computationally and requires no training data. It successfully identified simple capitalized entities like 'Pierre' and 'Vinken'.
   - Weaknesses: It relies on rigid, naive capitalization rules. It is highly prone to False Positives (e.g., misclassifying the first word of a sentence as an entity) and cannot classify the specific *type* of entity (Person vs. Organization).

2. Statistical Approach (spaCy):
   - Strengths: Uses deep learning representations trained on massive datasets like OntoNotes or CONLL-2003. It accurately understands context, properly classifying 'Elsevier N.V.' as an ORG and 'Pierre Vinken' as a PERSON. It is significantly more robust against noisy text and casing errors.
   - Weaknesses: Requires heavier computational overhead and memory to load the pre-trained neural network.

Conclusion: 
The statistical model vastly outperforms the rule-based approach in precision and utility. Natural language is inherently ambiguous, and regex cannot distinguish between "Apple" (a fruit starting a sentence) and "Apple" (the organization). SpaCy evaluates surrounding contextual vectors to make highly accurate, classified predictions.
"""
print(justification)

========== NAMED ENTITY RECOGNITION (NER) ==========
Loading NLTK Treebank corpus...

Raw Text for Analysis:
Pierre Vinken , 61 years old , will join the board as a nonexecutive director Nov. 29 . Mr. Vinken is chairman of Elsevier N.V. , the Dutch publishing group . Rudolph Agnew , 55 years old and former chairman of Consolidated Gold Fields PLC , was named *-1 a nonexecutive director of this British industrial conglomerate .

========== 1. RULE-BASED NER ==========
Entities extracted via Rule-Based (Regex):
['Pierre', 'Vinken', 'Nov.', 'Vinken', 'Elsevier', 'N.V.', 'Dutch', 'Rudolph', 'Agnew', 'Consolidated', 'Gold', 'Fields', 'PLC', 'British']

========== 2. STATISTICAL NER (spaCy) ==========
Entities extracted via spaCy:
 - Pierre Vinken (Class: PERSON)
 - 61 years old (Class: DATE)
 - Nov. 29 (Class: DATE)
 - Vinken (Class: PERSON)
 - Elsevier N.V. (Class: ORG)
 - Dutch (Class: NORP)
 - Rudolph Agnew (Class: PERSON)
 - 55 years old (Class: DATE)
 - Consolidated Gold Fields PLC (Cl